In [ ]:
import pandas as pd
import glob

# Load all CSV files matching the pattern
files = glob.glob("data/pl*.csv")

# Read each file into a dataframe and store in a list
dfs = []
for file in files:
    df = pd.read_csv(file)
    rename_map = {
        'BbAvH':'AvgH',
        'BbAvD':'AvgD',
        'BbAvA':'AvgA',
        'BbMxH':'MaxH',
        'BbMxD':'MaxD',
        'BbMxA':'MaxA',
    }
    df.columns = df.columns.str.replace(' ', '')
    df = df.rename(columns=rename_map)
    df['SourceFile'] = file 
    dfs.append(df)

# Find the intersection of columns across all dataframes
common_cols = set(dfs[0].columns)
for df in dfs[1:]:
    common_cols.intersection_update(set(df.columns))

reference_columns = dfs[-1].columns
final_ordered_cols = [c for c in reference_columns if c in common_cols]
# Reorder columns in each dataframe to match the reference
df_final_list = []
for df in dfs:
    df_final_list.append(df[final_ordered_cols])

# Concatenate all dataframes into a single dataframe
data = pd.concat(df_final_list, ignore_index=True)
print(f"✅ Loaded {len(data)} matches from {len(files)} files.")
data.head(20)


In [ ]:
# ==========================================
# 1 Handle missing/invalid Date values
# ==========================================

# Create 'RawDate' column if it doesn't exist
if 'RawDate' not in data.columns:
    data['RawDate'] = data['Date'].astype(str)

# Trim whitespace from 'RawDate'
data['RawDate'] = data['RawDate'].astype(str).str.strip()

# Define invalid tokens
_invalid_tokens = {'', 'na', 'nan', 'none', 'null', 'NaT'}

# Identify missing/invalid Date entries
mask_invalid_date = (
    data['RawDate'].isna() |
    data['RawDate'].str.lower().isin(_invalid_tokens)
)

# Report and handle invalid dates
missing_cnt = mask_invalid_date.sum()
print(f"⚠️  Found {missing_cnt} rows with missing/invalid Date")

if missing_cnt > 0:
    # Backup these rows for reference
    invalid_rows = data.loc[mask_invalid_date]
    invalid_rows.to_csv("invalid_date_rows.csv", index=False)
    print("💾 Saved invalid date rows to 'invalid_date_rows.csv' for reference.")
    
    # Remove invalid rows from main data
    data = data.loc[~mask_invalid_date].reset_index(drop=True)
    print(f"✅ Removed {missing_cnt} invalid rows. Remaining rows: {len(data)}")
else:
    print("✅ No invalid Date values found.")

styles = [
    dict(selector="th", props=[("font-size", "12px"), ("text-align", "center"), ("background-color", "#5656E6"), ("color", "white")]),
    dict(selector="td", props=[("text-align", "center")]),
    dict(selector="tr:hover", props=[("background-color", "#f0f0f0")]) 
]

(df.head(10).style
    .set_table_styles(styles)
    .format("{:.0f}", subset=['FTHG', 'FTAG'])
)


In [ ]:
# Remove columns that are partially empty
data = data.dropna(axis=1, how='any')
print(f"✅ Columns after removing empty ones: {len(data.columns)}")
print(f"Remaining rows: {len(data)}")

# Summary of data types, missing values, and unique counts
summary = pd.DataFrame({
    'dtype': data.dtypes,
    'missing_count': data.isna().sum(),
    'unique_count': data.nunique()
})
summary



In [ ]:
# ================================
# 2 ️ Standardize date formats
# ================================

from datetime import datetime

data['RawDate'] = data['Date'].astype(str).str.strip()

def try_parse_date(x):
    """
    Tries to parse a date string from multiple common EPL formats.
    Returns a datetime object or NaT if no format matches.
    """
    if pd.isna(x) or str(x).strip() == "":
        return pd.NaT
    
    x = str(x).strip()
    date_formats = [
        "%d/%m/%Y",  # 13/08/2011
        "%d/%m/%y",  # 13/08/11
        "%Y-%m-%d",  # 2011-08-13
        "%d-%m-%Y",  # 13-08-2011
        "%d-%m-%y",  # 13-08-11
        "%d.%m.%Y",  # 13.08.2011
        "%d.%m.%y",  # 13.08.11
        "%b %d %Y",  # Aug 13 2011
        "%d %b %Y",  # 13 Aug 2011
        "%B %d %Y",  # August 13 2011
        "%d %B %Y",  # 13 August 2011
    ]
    
    for fmt in date_formats:
        try:
            return datetime.strptime(x, fmt)
        except:
            continue
    return pd.NaT

# First attempt with dayfirst=True
data['Date'] = pd.to_datetime(data['RawDate'], dayfirst=True, errors='coerce')

# Second attempt with dayfirst=False for unparsed dates
mask = data['Date'].isna()
data.loc[mask, 'Date'] = data.loc[mask, 'RawDate'].apply(try_parse_date)

# check unparsed dates
unparsed = data[data['Date'].isna()]
print(f"⚠️  Unparsed dates remaining: {len(unparsed)}")

if len(unparsed) > 0:
    print("Here are a few examples of unparsed date formats:")
    display(unparsed[['SourceFile', 'RawDate']].head(10))
    unparsed[['SourceFile', 'RawDate']].to_csv('unparsed_dates_check.csv', index=False)
    print("💾 Saved unparsed dates to 'unparsed_dates_check.csv' for manual review.")

print("✅ Date parsing completed.")
print("Earliest date:", data['Date'].min())
print("Latest date:", data['Date'].max())
print("Remaining rows:", len(data))
data.head(20)



In [ ]:
# ================================
# 3 ️ Final cleanup and sorting
# ================================

# Ensure 'Date' column is datetime type
data['Date'] = pd.to_datetime(data['Date'], errors='coerce')

# Remove time part, keep only date
data['Date'] = data['Date'].dt.date

data = data.sort_values('Date').reset_index(drop=True)

print("Remaining rows:", len(data))
print("✅ Data sorted by date.")

display(data['Date'].head(5))
display(data['Date'].tail(5))


In [ ]:
# ================================
# 4 ️ Add Season column and summarize
# ================================

data['Season'] = data['Date'].apply(
    lambda d: d.year if d.month >= 8 else d.year - 1
)

print("✅ Season column added based on July cut-off rule.")
data[['Date', 'Season']].head(10)

# calculate match counts per season
season_match_counts = data['Season'].value_counts().sort_index()

print("📊 Matches per Season:")
print(season_match_counts)

# Create a summary dataframe
season_summary = season_match_counts.reset_index()
season_summary.columns = ['Season', 'MatchCount']

In [ ]:
# ================================
# 5 ️ Display cleaned data
# ================================

# Save cleaned data to CSV
data.to_csv("cleaned_data.csv", index=False)
# validate saved file
saved_data = pd.read_csv("cleaned_data.csv")
print("Saved data preview:")
display(saved_data.head(30))

